<a href="https://colab.research.google.com/github/youssef-mm/FlyRank-ML-Assignment/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/youssef-mm/FlyRank-ML-Assignment/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task Type:** Binary Classification with a downstream Ranking / Scoring application.

**Framing:**
We frame the Content Refresh lane as a **Binary Classification** task to predict whether a given content page is in an active state of performance decline (`is_declining_label`).

The model's calibrated probability outputs are subsequently combined with demand and visibility metrics to produce a **Ranked Review Queue**. This directly matches editorial workflows: content teams operate with a fixed weekly budget (e.g., reviewing 20 to 50 pages), so predicting binary probabilities allows us to score and rank candidate pages in order of refresh urgency and traffic exposure.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

### Target / Proxy Definition
- **The Label:** `is_declining_label`, defined as `trend_direction == "down"`.
- **Label Origin:** In this starter dataset, this is an **observable proxy label** calculated by comparing search impressions over the most recent 30 days against the preceding 30 days (`trend_pct < -20%`).
- **Honest Proxy Nuance:** It is important to acknowledge that this starter target is a comparative trailing-window proxy, not a prospective future outcome. In the full warehouse setup, this proxy is upgraded to a forward-looking window (e.g., prior 90 days of features predicting the subsequent 30 days of performance).
- **Target Leakage Discipline:** Because `is_declining_label` is mathematically derived from `trend_pct` and `trend_direction`, neither `trend_pct` nor `trend_direction` can ever be included in the feature set.

In [1]:
import os
import pandas as pd

# Load dataset (local path with Colab raw URL fallback)
data_path = "data/raw/content_refresh_anonymized.csv"
if not os.path.exists(data_path):
    data_path = "../../data/raw/content_refresh_anonymized.csv"
if not os.path.exists(data_path):
    data_path = "https://raw.githubusercontent.com/youssef-mm/FlyRank-ML-Assignment/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(data_path)

# Sketch the target column as a binary integer (1 = declining, 0 = stable/up)
df['target_is_declining'] = (df['trend_direction'] == 'down').astype(int)

total_pages = len(df)
declining_count = df['target_is_declining'].sum()
base_rate_pct = df['target_is_declining'].mean() * 100

print(f"Total pages in dataset: {total_pages:,}")
print(f"Target distribution ('down' = 1, other = 0):")
print(f" - Declining pages (1): {declining_count:,} ({base_rate_pct:.2f}% base rate)")
print(f" - Stable/Growing pages (0): {total_pages - declining_count:,} ({100 - base_rate_pct:.2f}%)")

print("\nTarget Column Sketch (Preview):")
preview_cols = ['content_id', 'client_id', 'impressions_90d', 'trend_direction', 'target_is_declining']
print(df[preview_cols].head())


Total pages in dataset: 30,000
Target distribution ('down' = 1, other = 0):
 - Declining pages (1): 16,262 (54.21% base rate)
 - Stable/Growing pages (0): 13,738 (45.79%)

Target Column Sketch (Preview):
             content_id          client_id  ...  trend_direction target_is_declining
0  content_304f48230142  client_f369cb89fc  ...             down                   1
1  content_a1fb4e703a9e  client_4e07408562  ...             down                   1
2  content_9aa793d4d895  client_7f2253d7e2  ...             down                   1
3  content_331d6c4de07b  client_19581e27de  ...           stable                   0
4  content_d99b7a2d90ca  client_3fdba35f04  ...             down                   1

[5 rows x 5 columns]


## 3. Success metric

*One metric you can defend. What number means 'good'?*

### Primary Metric: Precision@K (specifically Precision@50 and Precision@20)
- **Why Precision@K:** Standard accuracy is deceptive on ranking tasks because predicting the majority class trivially yields 54.2% accuracy without providing any useful prioritization. Editorial teams have finite weekly capacity (e.g., reviewing 20 to 50 pages per week). Precision@K measures the exact fraction of the top $K$ recommended pages that are genuinely in decline, directly preventing wasted editorial effort.
- **Supporting Metric:** **ROC-AUC** (to evaluate global ranking quality across all thresholds).

### What Number Means 'Good'?
- **Unconditioned Base Rate:** **54.2%** (a random sample yields ~27 true positives out of 50).
- **Uncalibrated Baseline Rule:** The starter heuristic baseline achieves **Precision@50 = 0.240 (24.0%)** (capturing only 12 of 50 pages because it selects low-volume dormant pages).
- **ML Success Threshold:** An ML model earns its place if it achieves **Precision@50 >= 0.700 (70.0%)** (at least 35 out of 50 pages correctly identified) and **ROC-AUC >= 0.750**, delivering substantial operational uplift over simple rules.

In [2]:
# Simulating the operational value of Precision@50 across methods
k = 50
base_rate = df['target_is_declining'].mean()
random_positives = int(k * base_rate)
baseline_rule_positives = int(k * 0.240)   # 12 of 50 (from starter baseline rule)
target_ml_positives = int(k * 0.740)       # 37 of 50 (from starter random forest benchmark)

print(f"Success Metric Evaluation (Editorial Capacity K = {k} pages):")
print(f" - Base Rate Expectation: Precision@{k} = {base_rate:.1%} (~{random_positives} true positives)")
print(f" - Heuristic Baseline Rule: Precision@{k} = 0.240 ({baseline_rule_positives} true positives)")
print(f" - Target ML Benchmark: Precision@{k} >= 0.700 ({target_ml_positives}+ true positives)")
print(f"\nOperational Impact: An honest ML model rescues editorial bandwidth, ensuring ~37 out of {k}")
print(f"reviewed pages truly need a refresh, vs only 12 under the unconditioned baseline rule.")


Success Metric Evaluation (Editorial Capacity K = 50 pages):
 - Base Rate Expectation: Precision@50 = 54.2% (~27 true positives)
 - Heuristic Baseline Rule: Precision@50 = 0.240 (12 true positives)
 - Target ML Benchmark: Precision@50 >= 0.700 (37+ true positives)

Operational Impact: An honest ML model rescues editorial bandwidth, ensuring ~37 out of 50
reviewed pages truly need a refresh, vs only 12 under the unconditioned baseline rule.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

### Unit of Analysis
**One row = One unique content page (`content_id`) for a specific client (`client_id`) evaluated over a trailing 90-day observation window.**

The slice includes metadata (`content_age_days`, `days_since_last_update`, `word_count`), search demand (`search_volume`, `competition`), Google Search Console performance (`impressions_90d`, `clicks_90d`, `avg_position`), and engagement signals (`sessions_90d`, `engagement_rate`).

In [3]:
print(f"Dataset Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Unique content_ids: {df['content_id'].nunique():,} (Grain check: exactly 1 row = 1 unique content item)")
print(f"Unique clients: {df['client_id'].nunique()}")

print("\nUnit of Analysis Preview (1 row = 1 Content Item):")
slice_cols = ['content_id', 'client_id', 'search_volume', 'impressions_90d', 'avg_position', 'days_since_last_update', 'target_is_declining']
print(df[slice_cols].head())


Dataset Shape: 30,000 rows x 45 columns
Unique content_ids: 30,000 (Grain check: exactly 1 row = 1 unique content item)
Unique clients: 32

Unit of Analysis Preview (1 row = 1 Content Item):
             content_id  ... target_is_declining
0  content_304f48230142  ...                   1
1  content_a1fb4e703a9e  ...                   1
2  content_9aa793d4d895  ...                   1
3  content_331d6c4de07b  ...                   0
4  content_d99b7a2d90ca  ...                   1

[5 rows x 7 columns]


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

### Why a Fixed Rule (If-Statement) Fails:
A naive heuristic rule (such as *"flag any page for refresh if days_since_last_update >= 180"*) fails because content performance decay is non-linear and conditional on search exposure:
1. **Evergreen Resilience:** High-authority evergreen articles can retain rankings and positive traffic for years without manual updates.
2. **Competitive Volatility:** New content in competitive topics can experience decay in under 60 days if competitors publish superior resources.
3. **The "Zombie Page" Phenomenon:** As demonstrated in the code below, pages unrefreshed for 181+ days actually show a *lower* decline rate (47.13%) than 91-180 day pages (61.11%). This is because unconditioned 181+ day pages are dormant "zombie" pages with near-zero baseline traffic (median impressions: 15.5) that have already bottomed out. A fixed rule wastes editorial budget rewriting dead pages.

### Where ML Wins:
Machine learning captures non-linear interactions across freshness, ranking position tiers, and search demand floors, prioritizing pages where an editorial refresh yields the highest traffic recovery.

In [4]:
# Demonstrating non-linear behavior across freshness tiers
decline_by_freshness = df.groupby('freshness_tier').agg(
    total_pages=('content_id', 'count'),
    declining_count=('target_is_declining', 'sum'),
    decline_rate_pct=('target_is_declining', lambda x: round(x.mean() * 100, 2)),
    median_impressions=('impressions_90d', 'median')
).reset_index()

print("Decline rate (%) and median impressions across content freshness tiers:")
print(decline_by_freshness.to_string(index=False))


Decline rate (%) and median impressions across content freshness tiers:
freshness_tier  total_pages  declining_count  decline_rate_pct  median_impressions
          0-30        20480            10473             51.14               470.0
          181+          174               82             47.13                15.5
         31-90          175              103             58.86               510.0
        91-180         9171             5604             61.11              1692.0


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.